In [1]:
import pandas as pd
import os
import numpy as np
import re

In [2]:
pd.set_option('display.max_columns', None)

# <span style="color:blue;">**Control**</span>

In [3]:
study = "Control"

## **STEP 0: Data preparation**

### 1. DICOM

#### Visit [parameter exlanation][peid] for more information

[peid]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/Doc.aspx?sourcedoc=%7B7B05A758-0D43-40AF-9177-E9C7522D2649%7D&file=Notes_parameters%20explaination.docx&action=default&mobileredirect=true 


In [4]:
# file_path = "P:/Dataset/R01-MO-DBT/MO-DBT-data-curation/dicom_tag.xlsx"
file_path = "/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data/dicom_tag.xlsx"
dicom = pd.read_excel(file_path)

In [5]:
dicom.rename(columns={'PatientID': 'PATIENT_STUDY_ID', 'AccessionNumber': 'ACCESSION_NUMBER'}, inplace=True)

In [6]:
dicom["PATIENT_STUDY_ID"].unique().size

5655

In [7]:
PIDs = dicom["PATIENT_STUDY_ID"].unique()

In [8]:
dicom.head(5)

,PATIENT_STUDY_ID,PatientBirthDate,PatientAge,ACCESSION_NUMBER,StudyDate,Study,Side,Series,View,Slab,StudyDescription,SeriesDescription,SeriesNumber,Manufacturer,ManufacturerModelName,SliceThickness,Exposure,Rows,Columns,PixelSpacing,FolderPath
0,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,NaN,SECURE,NaN,NaN,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,Hologic R2 ImageChecker CAD SC,1,"R2 Technology, Inc.",Cenova,NaN,NaN,1500.0,1250.0,NaN,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
1,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,R,FFDM,ML,NaN,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,R ML,71100000,"HOLOGIC, Inc.",Selenia Dimensions,NaN,102.0,3328.0,2560.0,0.038889\0.038889,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
2,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,R,FFDM,XCCL,NaN,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,R XCCL,71100000,"HOLOGIC, Inc.",Selenia Dimensions,NaN,94.0,3328.0,2560.0,0.038889\0.038889,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
3,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,L,C VIEW,LM,NaN,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,L LM C-View,71300000,"HOLOGIC, Inc.",Selenia Dimensions,NaN,71.0,2457.0,1890.0,0.087290\0.087290,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
4,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,L,C VIEW,MLO,NaN,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,L MLO C-View,71300000,"HOLOGIC, Inc.",Selenia Dimensions,NaN,87.0,2457.0,1890.0,0.086609\0.086609,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...


#### Create a list of unique study visit to serve as the reference linking the EHR to the images available for specific patient visits (as images are limited to certain visits, not all).

#### See shared parameters in [parameter exlanation][peid] to link data

[peid]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/Doc.aspx?sourcedoc=%7B7B05A758-0D43-40AF-9177-E9C7522D2649%7D&file=Notes_parameters%20explaination.docx&action=default&mobileredirect=true 

In [9]:
# Define the key identifier columns and columns to extract
key_columns = ['PATIENT_STUDY_ID', 'PatientAge', 'ACCESSION_NUMBER', 'StudyDate', 'Study', 'Side', 'Slab']
unique_study = dicom.drop_duplicates(subset=key_columns, keep='first')

unique_study = unique_study[key_columns]
unique_study = unique_study[unique_study['Side'].notna()]

unique_study.reset_index(drop=True, inplace=True)

In [10]:
unique_study.head(5)

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,Side,Slab
0,4330018595,54.0,63737104,2019-08-19,DIAG,R,NaN
1,4330018595,54.0,63737104,2019-08-19,DIAG,L,NaN
2,4330018595,54.0,60103700,2020-06-02,DIAG,R,NaN
3,4330018595,54.0,60690108,2020-06-02,SCREEN,L,NaN
4,4330029102,44.0,64888584,2019-05-16,SCREEN,L,NaN


In [11]:
unique_study["PATIENT_STUDY_ID"].nunique()

5650

#### Filter study with DBT 

In [12]:
dicom_dbt = dicom[dicom["Series"]=="DBT"]

key_columns = ['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'StudyDate', 'Study', 'Side', 'Slab']
unique_study_dicom_dbt = dicom_dbt.drop_duplicates(subset=key_columns, keep='first')
unique_study_dicom_dbt.reset_index(drop=True, inplace=True)

In [13]:
unique_study_dicom_dbt.head(5)

,PATIENT_STUDY_ID,PatientBirthDate,PatientAge,ACCESSION_NUMBER,StudyDate,Study,Side,Series,View,Slab,StudyDescription,SeriesDescription,SeriesNumber,Manufacturer,ManufacturerModelName,SliceThickness,Exposure,Rows,Columns,PixelSpacing,FolderPath
0,4330066079,1989-07-01,30.0,61259016,2020-01-14,DIAG,L,DBT,CC,NaN,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,L CC Breast Tomosynthesis Image,73200000,"HOLOGIC, Inc.",Selenia Dimensions,1.0,NaN,2457.0,1890.0,0.088364\0.088364,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
1,4330066079,1989-07-01,30.0,61259016,2020-01-14,DIAG,R,DBT,CC,NaN,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,R CC Breast Tomosynthesis Image,73200000,"HOLOGIC, Inc.",Selenia Dimensions,1.0,NaN,2457.0,1890.0,0.088500\0.088500,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
2,4330116791,1963-07-01,56.0,61499674,2019-12-17,DIAG,L,DBT,CC,NaN,DIAG DIG MAMMO LEFT ALL VIEWS WITH TOMOSYNTHESIS,L CC Breast Tomosynthesis Image,73200000,"HOLOGIC, Inc.",Selenia Dimensions,1.0,NaN,2457.0,1996.0,0.106407\0.106407,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
3,4330313855,1980-07-01,37.0,77236131,2018-05-30,DIAG,L,DBT,CC,NaN,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,L CC Breast Tomosynthesis Image,73200000,"HOLOGIC, Inc.",Selenia Dimensions,1.0,NaN,2457.0,1890.0,0.088048\0.088048,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
4,4330313855,1980-07-01,37.0,77236131,2018-05-30,DIAG,R,DBT,CC,NaN,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,R CC Breast Tomosynthesis Image,73200000,"HOLOGIC, Inc.",Selenia Dimensions,1.0,NaN,2457.0,1890.0,0.087911\0.087911,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...


In [14]:
unique_study_dicom_dbt["PATIENT_STUDY_ID"].nunique()

2433

In [15]:
unique_patient_dbt = unique_study[unique_study["PATIENT_STUDY_ID"].isin(unique_study_dicom_dbt["PATIENT_STUDY_ID"].unique())]

In [16]:
unique_patient_dbt # unique patient that has dbt across stidues

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,Side,Slab
8,4330066079,30.0,61259016,2020-01-14,DIAG,L,NaN
9,4330066079,30.0,61259016,2020-01-14,DIAG,R,NaN
15,4330116791,56.0,62335422,2019-09-24,DIAG,L,NaN
16,4330116791,56.0,61499674,2019-12-17,DIAG,L,NaN
77,4330313855,37.0,77236131,2018-05-30,DIAG,L,NaN
...,...,...,...,...,...,...,...
32486,4339601084,46.0,77038224,2018-11-07,DIAG,L,NaN
32487,4339601084,46.0,77038224,2018-11-07,DIAG,R,NaN
32488,4339601084,46.0,65758752,2019-05-16,DIAG,R,NaN
32489,4339601084,47.0,60049293,2020-04-09,SCREEN,L,NaN


In [17]:
unique_patient_dbt = pd.merge(
    unique_patient_dbt,
    unique_study_dicom_dbt[['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'StudyDate', 'Study', 'Side', 'Series', 'Slab']],
    on = key_columns,
    how='left'
)

In [18]:
unique_patient_dbt

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,Side,Slab,Series
0,4330066079,30.0,61259016,2020-01-14,DIAG,L,NaN,DBT
1,4330066079,30.0,61259016,2020-01-14,DIAG,R,NaN,DBT
2,4330116791,56.0,62335422,2019-09-24,DIAG,L,NaN,NaN
3,4330116791,56.0,61499674,2019-12-17,DIAG,L,NaN,DBT
4,4330313855,37.0,77236131,2018-05-30,DIAG,L,NaN,DBT
...,...,...,...,...,...,...,...,...
18349,4339601084,46.0,77038224,2018-11-07,DIAG,L,NaN,DBT
18350,4339601084,46.0,77038224,2018-11-07,DIAG,R,NaN,DBT
18351,4339601084,46.0,65758752,2019-05-16,DIAG,R,NaN,DBT
18352,4339601084,47.0,60049293,2020-04-09,SCREEN,L,NaN,NaN


### 2. Electric Health Record

#### Visit [parameter exlanation][peid] for more information

[peid]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/Doc.aspx?sourcedoc=%7B7B05A758-0D43-40AF-9177-E9C7522D2649%7D&file=Notes_parameters%20explaination.docx&action=default&mobileredirect=true 


In [19]:
# file_path = "P:/Dataset/R01-MO-DBT/MO-DBT-data-curation/parameters of interest.xlsx"
file_path = "/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data/parameters of interest.xlsx"
file_name = pd.ExcelFile(file_path).sheet_names
file_name

['enteredit_findings',
 'pathology',
 'pathology_findings',
 'patient_data_ie',
 'hormonal_mens',
 'risk_factors',
 'vitals',
 'patient_demo',
 'procedure_notes',
 'procedures']

---

# **Extract <span style="color:blue;">Control**</span> 

#### See [inclusion criteria][ic] to for filtering

[ic]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/doc.aspx?sourcedoc=%7B9c403413-a771-4830-98ab-15171872c4dd%7D&action=edit

## **STEP 1**. Filter patients (no pathology, BI-RADS = 1, 2) then merge with patients with DBT available
### (OUTPUT) control_cohort

enteredit_findings
* Columns: COMPOSITION_NAME, FINDING_CATEGORY, FINDING_REC, EXAM_COMPLETED_DATE
* Cancer: 
    * FINDING_CATEGORY = 1, 2

pathology
* Columns: BX_ID, PATHOLOGY_DATE, LESION_CLASS, SIDE
* Control: 
    * If pathology exists --> exclude

### <span style="color:#FF6347;">**READ**</span> file (EHR: enteredit_findings, pathology)

In [20]:
file_name

['enteredit_findings',
 'pathology',
 'pathology_findings',
 'patient_data_ie',
 'hormonal_mens',
 'risk_factors',
 'vitals',
 'patient_demo',
 'procedure_notes',
 'procedures']

In [21]:
fn = file_name[0]
# file_path = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, "Cleaned", fn + ".xlsx")
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, "Cleaned", fn + ".xlsx")
birads = pd.read_excel(file_path)

In [22]:
birads.head(3)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count
0,4330000534,78048053,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2018-03-28,2
1,4330000534,64787403,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2019-05-18,2
2,4330000534,63280481,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2019-06-20,2


In [23]:
fn = file_name[1]
# file_path = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, "Cleaned", fn + ".xlsx")
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, "Cleaned", fn + ".xlsx")
pathology = pd.read_excel(file_path)

In [24]:
pathology.head(3)

,PATIENT_STUDY_ID,BX_ID,PATHOLOGY_DATE,LESION_CLASS,SIDE
0,4330189702,418851,2019-04-26,Malignant,R
1,4330311096,450823,2016-11-10,Malignant,R
2,4330311096,459526,2016-11-28,Malignant,R


### **1. Filter out patients with pathology**

In [25]:
birads.drop(columns="FINDING_LOCATION", inplace=True)

In [26]:
# Identify all patient IDs that have pathology records
pathology_pids = pathology['PATIENT_STUDY_ID'].unique()

# Exclude any patient who appears in pathology at all
birads_no_pathology = birads[~birads['PATIENT_STUDY_ID'].isin(pathology_pids)].copy()

print(f"Patients (BI-RADS) : {birads['PATIENT_STUDY_ID'].nunique()}")
print(f"Patients removed (have pathology) : {birads['PATIENT_STUDY_ID'].nunique() - birads_no_pathology['PATIENT_STUDY_ID'].nunique()}")
print(f"Patients (BI-RADS, no pathology) : {birads_no_pathology['PATIENT_STUDY_ID'].nunique()}")

Patients (BI-RADS) : 38339
Patients removed (have pathology) : 362
Patients (BI-RADS, no pathology) : 37977


### **2. Filter BI-RADS findings for Control criteria** (BI-RADS 1 or 2)

In [27]:
# Keep only BI-RADS 1 (Negative) and BI-RADS 2 (Benign) findings
normal_categories = ['1 - Negative', '2 - Benign finding']

birads_control = birads_no_pathology[
    birads_no_pathology['FINDING_CATEGORY'].isin(normal_categories)
].copy()


birads_control.reset_index(inplace=True, drop=True)

print(f"Rows for BIRADS 1 or 2 filter      : {len(birads_control)}")

Rows for BIRADS 1 or 2 filter      : 234737


In [28]:
# Handle bilateral exams: no SIDE + duplicate_count ≥ 2 → expand to L and R
no_side  = birads_control.copy()

bilateral_mask = no_side['duplicate_count'] >= 2
bilateral      = no_side[bilateral_mask].copy()
rest           = no_side[~bilateral_mask].copy()

bilateral_L = bilateral.copy(); bilateral_L['SIDE'] = 'L'
bilateral_R = bilateral.copy(); bilateral_R['SIDE'] = 'R'

birads_control = pd.concat(
    [bilateral_L, bilateral_R, rest], ignore_index=True
)
print(f"birads_control             : {len(birads_control)}")

birads_control             : 415095


In [29]:
birads_control.sort_values(by=["PATIENT_STUDY_ID", "ACCESSION_NUMBER", "EXAM_COMPLETED_DATE"], inplace=True)
birads_control.reset_index(inplace=True, drop=True)

In [30]:
birads_control

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,SIDE
0,4330000534,60228678,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2020-06-01,1,NaN
1,4330000534,60687102,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2020-06-10,1,NaN
2,4330000534,63280481,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-06-20,2,L
3,4330000534,63280481,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-06-20,2,R
4,4330000534,67674008,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2021-06-03,2,L
...,...,...,...,...,...,...,...,...
415090,4339988199,65403287,Scattered fibroglandular (25% - 50%),1 - Negative,N-Normal interval follow-up,2018-11-28,2,R
415091,4339988199,67718778,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2021-02-25,2,L
415092,4339988199,67718778,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2021-02-25,2,R
415093,4339988199,452549960,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2022-03-01,2,L


### **3. Merge control DBT cohort with BI-RADS records**

In [31]:
unique_patient_dbt.columns

Index(['PATIENT_STUDY_ID', 'PatientAge', 'ACCESSION_NUMBER', 'StudyDate',
       'Study', 'Side', 'Slab', 'Series'],
      dtype='object')

In [32]:
birads_control.columns

Index(['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'COMPOSITION_NAME',
       'FINDING_CATEGORY', 'FINDING_REC', 'EXAM_COMPLETED_DATE',
       'duplicate_count', 'SIDE'],
      dtype='object')

In [33]:
# ── STEP 2: Connect to DICOM images (unique_patient_dbt) ─────────────────────

unique_patient_dbt = unique_patient_dbt.rename(columns={'Side': 'SIDE'})

has_side = (birads_control[birads_control['SIDE'].notna()]
            .drop_duplicates(subset=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'SIDE'], keep='first'))
no_side  = (birads_control[birads_control['SIDE'].isna()]
            .drop_duplicates(subset=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'], keep='first')
            .drop(columns='SIDE'))

# 1. Sided match
merge_sided = pd.merge(
    unique_patient_dbt, has_side,
    on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'SIDE'], how='inner'
)

# 2. No-side match — only DICOM rows whose accession wasn't already matched above
sided_accessions = merge_sided[['PATIENT_STUDY_ID', 'ACCESSION_NUMBER']].drop_duplicates()
dicom_unmatched  = unique_patient_dbt.merge(
    sided_accessions, on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'],
    how='left', indicator=True
).query('_merge == "left_only"').drop(columns='_merge')

merge_no_side = pd.merge(
    dicom_unmatched, no_side,
    on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'], how='inner'
)

# 3. DICOM rows with no birads match at all → keep with NaN birads columns
matched_keys = pd.concat([
    merge_sided[['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'SIDE']],
    merge_no_side[['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'SIDE']]
]).drop_duplicates()

no_birads = unique_patient_dbt.merge(
    matched_keys, on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'SIDE'],
    how='left', indicator=True
).query('_merge == "left_only"').drop(columns='_merge')

df_final = pd.concat([merge_sided, merge_no_side, no_birads], ignore_index=True)
df_final = df_final.sort_values(['PATIENT_STUDY_ID', 'StudyDate', 'ACCESSION_NUMBER', 'SIDE'], ignore_index=True).drop(columns=['duplicate_count'])

print(f"unique_patient_dbt : {unique_patient_dbt.shape[0]} rows")
print(f"  ↳ with side      : {len(merge_sided)}")
print(f"  ↳ without side   : {len(merge_no_side)}")
print(f"  ↳ no birads      : {len(no_birads)}")
print(f"df_final           : {df_final.shape[0]} rows")

unique_patient_dbt : 18354 rows
  ↳ with side      : 8
  ↳ without side   : 10
  ↳ no birads      : 18336
df_final           : 18354 rows


In [34]:
unique_patient_dbt[unique_patient_dbt['PATIENT_STUDY_ID']==4333577909]

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,SIDE,Slab,Series
11634,4333577909,59.0,78312102,2018-04-17,NaN,L,NaN,NaN
11635,4333577909,59.0,78312204,2018-04-17,DIAG,L,NaN,DBT
11636,4333577909,59.0,78312204,2018-04-17,DIAG,R,NaN,DBT
11637,4333577909,61.0,64218975,2019-07-08,SCREEN,L,NaN,DBT
11638,4333577909,64.0,451709002,2022-10-10,SCREEN,L,Y,DBT


In [35]:
birads[birads['PATIENT_STUDY_ID']==4333210516]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count
41624,4333210516,86998028,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2016-04-06,2
41625,4333210516,72939576,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2017-04-08,2
41626,4333210516,78114012,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-05-11,2
41627,4333210516,64957868,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-05-24,2
41628,4333210516,66178393,Heterogeneously dense (51% - 75%),0 - Need additional imaging evaluation,P-Additional projections,2021-04-21,1
41629,4333210516,66178393,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2021-04-21,1
41630,4333210516,66003423,Heterogeneously dense (51% - 75%),3 - Probably benign - short interval follow-up,F-Follow-up at short interval (1-11 months),2021-05-07,1
41631,4333210516,453496082,Heterogeneously dense (51% - 75%),3 - Probably benign - short interval follow-up,F-Follow-up at short interval (1-11 months),2021-12-03,2
41632,4333210516,453374936,Heterogeneously dense (51% - 75%),6 - Known biopsy proven malignancy,K-Appropriate action should be taken,2021-12-23,1
41633,4333210516,459318559,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2022-12-05,2


In [36]:
pathology[pathology['PATIENT_STUDY_ID']==4333210516]

,PATIENT_STUDY_ID,BX_ID,PATHOLOGY_DATE,LESION_CLASS,SIDE


In [37]:
df_final[df_final['PATIENT_STUDY_ID']==4333210516]

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,SIDE,Slab,Series,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE
6066,4333210516,77.0,72939576,2017-04-08,SCREEN,L,NaN,DBT,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2017-04-08
6067,4333210516,77.0,72939576,2017-04-08,SCREEN,R,NaN,DBT,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2017-04-08
6068,4333210516,78.0,78114012,2018-05-11,SCREEN,L,NaN,NaN,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-05-11
6069,4333210516,78.0,78114012,2018-05-11,SCREEN,R,NaN,NaN,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-05-11
6070,4333210516,79.0,64957868,2019-05-24,SCREEN,L,NaN,NaN,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-05-24
6071,4333210516,79.0,64957868,2019-05-24,SCREEN,R,NaN,NaN,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-05-24
6072,4333210516,81.0,66178393,2021-04-21,SCREEN,L,NaN,DBT,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2021-04-21
6073,4333210516,81.0,66178393,2021-04-21,SCREEN,R,NaN,DBT,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2021-04-21
6074,4333210516,82.0,453374936,2021-12-23,DIAG,R,Y,DBT,NaN,NaN,NaN,NaN
6075,4333210516,83.0,459318559,2022-12-05,DIAG,L,Y,DBT,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2022-12-05


In [38]:
print(df_final.shape, unique_patient_dbt.shape)

(18354, 12) (18354, 8)


### **4. Retain DBT cohort with BI-RADS 1 or 2 only**

In [39]:
# Keep only BI-RADS 1 (Negative) and BI-RADS 2 (Benign) findings
normal_categories = ['1 - Negative', '2 - Benign finding']

df_step1 = df_final[
    df_final['FINDING_CATEGORY'].isin(normal_categories)
].copy()


df_step1.reset_index(inplace=True, drop=True)

print(f"Rows for BIRADS 1 or 2 filter      : {len(df_step1)}")

Rows for BIRADS 1 or 2 filter      : 18


In [40]:
df_step1.shape

(18, 12)

### <span style="color:#FF6347;">**SAVE**</span> file

In [41]:
# output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'cancer_cohort' + ".xlsx")
output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, "control_cohort" + ".xlsx")
df_step1.to_excel(output_file, index=False)

## **STEP 2**. <span style="color:blue;">**Control**</span>: Label  "Index" = <span style="color:#8A2BE2;">**INDEX**</span> or <span style="color:#00BFFF;">**INDEX-1**</span>
### (OUTPUT) control_cohort

### <span style="color:#FF6347;">**READ**</span> file (control_cohort)

In [42]:
# output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'control_cohort' + ".xlsx")
output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, "control_cohort" + ".xlsx")
control_cohort = pd.read_excel(output_file)

In [43]:
control_cohort["EXAM_COMPLETED_DATE"] = pd.to_datetime(control_cohort["EXAM_COMPLETED_DATE"], format="%Y-%m-%d")

In [44]:
control_cohort["PATIENT_STUDY_ID"].nunique()

3

In [45]:
control_cohort.head(5)

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,SIDE,Slab,Series,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE
0,4333210516,77,72939576,2017-04-08,SCREEN,L,NaN,DBT,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2017-04-08
1,4333210516,77,72939576,2017-04-08,SCREEN,R,NaN,DBT,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2017-04-08
2,4333210516,78,78114012,2018-05-11,SCREEN,L,NaN,NaN,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-05-11
3,4333210516,78,78114012,2018-05-11,SCREEN,R,NaN,NaN,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-05-11
4,4333210516,79,64957868,2019-05-24,SCREEN,L,NaN,NaN,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-05-24


## 1. Locate <span style="color:blue;">**Control**</span> year, "Index" = <span style="color:#8A2BE2;">**INDEX**</span>

#### See [inclusion criteria][ic] to for filtering

[ic]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/doc.aspx?sourcedoc=%7B9c403413-a771-4830-98ab-15171872c4dd%7D&action=edit

In [46]:
cohort = control_cohort.copy()
cohort['Index'] = None
cohort['EXAM_COMPLETED_DATE'] = pd.to_datetime(cohort['EXAM_COMPLETED_DATE'])

In [47]:
# ── STEP 1: Locate INDEX ──────────────────────────────────────────────────────
# INDEX = latest SCREEN exam with BIRADS 1/2 and image available
# idx_screen = cohort['Study'] == 'SCREEN'
idx_normal = cohort['FINDING_CATEGORY'].isin(['1 - Negative', '2 - Benign finding'])

latest_index = (
    cohort[idx_normal]
    .groupby('PATIENT_STUDY_ID')['EXAM_COMPLETED_DATE']
    .max()                    # <── latest qualifying exam
    .reset_index()
    .rename(columns={'EXAM_COMPLETED_DATE': 'INDEX_DATE'})
)

cohort = cohort.merge(latest_index, on='PATIENT_STUDY_ID', how='left')

mask_index = cohort['EXAM_COMPLETED_DATE'] == cohort['INDEX_DATE']
cohort.loc[mask_index, 'Index'] = 'INDEX'

print(f"✅ INDEX rows    : {mask_index.sum()}")
print(f"   Patients      : {cohort[mask_index]['PATIENT_STUDY_ID'].nunique()}")

✅ INDEX rows    : 4
   Patients      : 3


## 2. Locate <span style="color:blue;">**Control with 1 year follow-up**</span> year, "Index" = <span style="color:#00BFFF;">**INDEX-1**</span>. Adjust <span style="color:pink;"> **screening interval (OR search period)**</span> 

Add threshold labeling for the same index time point (threshold ~1.5 months)

#### See [inclusion criteria][ic] to for filtering

[ic]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/doc.aspx?sourcedoc=%7B9c403413-a771-4830-98ab-15171872c4dd%7D&action=edit

In [48]:
# ── Parameters ────────────────────────────────────────────────────────────────
min_months      = pd.Timedelta(days=0)              # how far back to look (min)
max_months      = pd.Timedelta(days=365/12 * 30)   # how far back to look (max)
threshold_group = pd.Timedelta(days=365/12 * 1.5)  # 1.5-month grouping

In [49]:
cohort['exam_to_index_diff'] = cohort['INDEX_DATE'] - cohort['EXAM_COMPLETED_DATE']

mask_window = (
    cohort['exam_to_index_diff'].notna() &
    (cohort['exam_to_index_diff'] > min_months) &
    (cohort['exam_to_index_diff'] <= max_months)
)

def assign_index_labels(dates):
    dates_sorted = sorted(set(dates), reverse=True)  # newest → oldest
    labels = {}
    rank = 0
    group_anchor = None
    for d in dates_sorted:
        if group_anchor is None or (group_anchor - d) > threshold_group:
            rank += 1
            group_anchor = d
        labels[d] = f'INDEX-{rank}'
    return labels

window_rows = cohort[mask_window].copy()
date_label_map = {}
for pid, grp in window_rows.groupby('PATIENT_STUDY_ID'):
    mapping = assign_index_labels(grp['EXAM_COMPLETED_DATE'])
    for date, label in mapping.items():
        date_label_map[(pid, date)] = label

cohort.loc[mask_window, 'Index'] = cohort.loc[mask_window].apply(
    lambda row: date_label_map.get(
        (row['PATIENT_STUDY_ID'], row['EXAM_COMPLETED_DATE']),
        row['Index']
    ),
    axis=1
)

print(cohort['Index'].value_counts(dropna=False))

Index
None       10
INDEX       4
INDEX-1     3
INDEX-2     1
Name: count, dtype: int64


In [50]:
# ── STEP 3: time_to_INDEX ─────────────────────────────────────────────────────
is_prior = cohort['Index'].str.match(r'^INDEX-\d+$', na=False)

cohort['time_to_INDEX_days'] = (
    cohort['INDEX_DATE'] - cohort['EXAM_COMPLETED_DATE']
).dt.days

cohort.loc[~is_prior, 'time_to_INDEX_days'] = pd.NA
cohort['time_to_INDEX_months'] = (cohort['time_to_INDEX_days'] / 30.44).round(1)

cohort = cohort.drop(columns=['INDEX_DATE', 'exam_to_index_diff'])

In [51]:
cohort.head(5)

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,SIDE,Slab,Series,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,Index,time_to_INDEX_days,time_to_INDEX_months
0,4333210516,77,72939576,2017-04-08,SCREEN,L,NaN,DBT,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2017-04-08,None,NaN,NaN
1,4333210516,77,72939576,2017-04-08,SCREEN,R,NaN,DBT,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2017-04-08,None,NaN,NaN
2,4333210516,78,78114012,2018-05-11,SCREEN,L,NaN,NaN,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-05-11,None,NaN,NaN
3,4333210516,78,78114012,2018-05-11,SCREEN,R,NaN,NaN,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-05-11,None,NaN,NaN
4,4333210516,79,64957868,2019-05-24,SCREEN,L,NaN,NaN,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-05-24,None,NaN,NaN


### <span style="color:#FF6347;">**SAVE**</span> file (control_cohort)

In [52]:
# output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'control_cohort' + ".xlsx")
output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, 'control_cohort' + ".xlsx")
cohort.to_excel(output_file, index=False)

In [53]:
cohort["PATIENT_STUDY_ID"].nunique()

3

## **STEP 3**. Extract <span style="color:blue;">**Normal**</span> cohort only
### (Output) normal, normal_cohort

#### <span style="color:#FF6347;">**READ**</span> file (control_cohort)

In [54]:
# output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'control_cohort' + ".xlsx")
output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, 'control_cohort' + ".xlsx")
cohort = pd.read_excel(output_file)

In [55]:
cohort.head(5)

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,SIDE,Slab,Series,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,Index,time_to_INDEX_days,time_to_INDEX_months
0,4333210516,77,72939576,2017-04-08,SCREEN,L,NaN,DBT,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2017-04-08,NaN,NaN,NaN
1,4333210516,77,72939576,2017-04-08,SCREEN,R,NaN,DBT,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2017-04-08,NaN,NaN,NaN
2,4333210516,78,78114012,2018-05-11,SCREEN,L,NaN,NaN,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-05-11,NaN,NaN,NaN
3,4333210516,78,78114012,2018-05-11,SCREEN,R,NaN,NaN,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-05-11,NaN,NaN,NaN
4,4333210516,79,64957868,2019-05-24,SCREEN,L,NaN,NaN,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-05-24,NaN,NaN,NaN


In [56]:
cohort["EXAM_COMPLETED_DATE"] = pd.to_datetime(cohort["EXAM_COMPLETED_DATE"], format="%Y-%m-%d")

### **Criteria**:
#### 1. No pathology ever
* Already met, no need to check
#### 2. At <span style="color:#8A2BE2;">**INDEX**</span>, "FINDING_CATEGORY" = "1 - Negative", "2 - Benign finding"
#### 3. At <span style="color:#8A2BE2;">**INDEX**</span>, "Study" = "SCREEN"
#### 4. At <span style="color:#00BFFF;">**INDEX-1**</span>, "Series" = "DBT"
#### 5. At <span style="color:#00BFFF;">**INDEX-1**</span>, "FINDING_CATEGORY" = "1 - Negative", "2 - Benign finding"
#### 6. At <span style="color:#00BFFF;">**INDEX-1**</span>, "Study" = "SCREEN"

In [57]:
normal_cats = ['1 - Negative', '2 - Benign finding']

recall_cats = [
    '0 - Need additional imaging evaluation',
    '3 - Probably benign - short interval follow-up',
    '4 - Suspicious abnormality, biopsy should be considered',
    '4A - Suspicious abnormality - biopsy should be considered - low suspicion',
    '4B - Suspicious abnormality - biopsy should be considered - intermediate suspicion',
    '4C - Suspicious abnormality - biopsy should be considered - moderate suspicion',
    '5 - Highly suggestive of malignancy, appropriate action should be taken',
    '6 - Known biopsy proven malignancy'
]

In [58]:
# Patients with ANY recall finding at INDEX-1 (disqualify, takes priority)
recall_patients = set(cohort.loc[
    ((cohort['Index'] == 'INDEX-1') | (cohort['Index'] == 'INDEX')) &
    (cohort['Series'] == 'DBT') &
    (cohort['FINDING_CATEGORY'].isin(recall_cats)) &
    (cohort['Study'] == 'SCREEN'),
    'PATIENT_STUDY_ID'
].unique())

# Patients with a normal finding at INDEX
normal_patients_at_index = set(cohort.loc[
    (cohort['Index'] == 'INDEX') &
    (cohort['FINDING_CATEGORY'].isin(normal_cats)) &
    (cohort['Study'] == 'SCREEN'),
    'PATIENT_STUDY_ID'
].unique())

# Patients with a normal finding at INDEX-1
normal_patients_at_index1 = set(cohort.loc[
    (cohort['Index'] == 'INDEX-1') &
    (cohort['Series'] == 'DBT') &
    (cohort['FINDING_CATEGORY'].isin(normal_cats)) &
    (cohort['Study'] == 'SCREEN'),
    'PATIENT_STUDY_ID'
].unique())

normal_patients = normal_patients_at_index & normal_patients_at_index1

# Exclude wins: remove any patient who appeared in recall
qualifying_patients = normal_patients - recall_patients

print(f"Normal patients     : {len(normal_patients)}")
print(f"Recall patients     : {len(recall_patients)}")
print(f"Overlap removed     : {len(normal_patients & recall_patients)}")
print(f"Qualifying patients : {len(qualifying_patients)}")

normal = cohort[cohort['PATIENT_STUDY_ID'].isin(qualifying_patients)].copy()
print(f"normal rows      : {len(normal)}")
print(f"Index breakdown:\n{normal['Index'].value_counts(dropna=False)}")

Normal patients     : 0
Recall patients     : 0
Overlap removed     : 0
Qualifying patients : 0
normal rows      : 0
Index breakdown:
Series([], Name: count, dtype: int64)


In [59]:
normal_cohort = cohort[cohort['PATIENT_STUDY_ID'].isin(qualifying_patients)].reset_index(drop=True)
normal_cohort.sort_values(['PATIENT_STUDY_ID', 'StudyDate', 'ACCESSION_NUMBER']).reset_index(drop=True, inplace=True)

normal_cohort["EXAM_COMPLETED_DATE"] = normal_cohort["EXAM_COMPLETED_DATE"].dt.strftime("%Y-%m-%d")

##### <span style="color:#FF6347;">**SAVE**</span> **normal_cohort**

In [60]:
normal_cohort['PATIENT_STUDY_ID'].nunique()

0

In [61]:
# output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'normal_cohort' + ".xlsx")
output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, 'normal_cohort' + ".xlsx")

normal_cohort.to_excel(output_file, index=False)

##### <span style="color:#FF6347;">**SAVE**</span> **normal** (only retain INDX & INDEX-1)

In [62]:
normal= normal_cohort[(normal_cohort["Index"]=="INDEX") | (normal_cohort["Index"]=="INDEX-1")]

In [63]:
normal['PATIENT_STUDY_ID'].nunique()

0

In [64]:
# output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'normal' + ".xlsx")
output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, 'normal' + ".xlsx")

normal.to_excel(output_file, index=False)

### **Broader Criteria**:
#### 1. No pathology ever
* Already met, no need to check
#### 2. At <span style="color:#8A2BE2;">**INDEX**</span>, "FINDING_CATEGORY" = "1 - Negative", "2 - Benign finding"
#### 4. At <span style="color:#00BFFF;">**INDEX-1**</span>, "Series" = "DBT"
#### 5. At <span style="color:#00BFFF;">**INDEX-1**</span>, "FINDING_CATEGORY" = "1 - Negative", "2 - Benign finding"

In [65]:
# Patients with ANY recall finding at INDEX-1 (disqualify, takes priority)
recall_patients = set(cohort.loc[
    ((cohort['Index'] == 'INDEX-1') | (cohort['Index'] == 'INDEX')) &
    (cohort['Series'] == 'DBT') &
    (cohort['FINDING_CATEGORY'].isin(recall_cats)),
    'PATIENT_STUDY_ID'
].unique())

# Patients with a normal finding at INDEX
normal_patients_at_index = set(cohort.loc[
    (cohort['Index'] == 'INDEX') &
    (cohort['FINDING_CATEGORY'].isin(normal_cats)),
    'PATIENT_STUDY_ID'
].unique())

# Patients with a normal finding at INDEX-1
normal_patients_at_index1 = set(cohort.loc[
    (cohort['Index'] == 'INDEX-1') &
    (cohort['Series'] == 'DBT') &
    (cohort['FINDING_CATEGORY'].isin(normal_cats)),
    'PATIENT_STUDY_ID'
].unique())

normal_patients = normal_patients_at_index & normal_patients_at_index1

# Exclude wins: remove any patient who appeared in recall
qualifying_patients = normal_patients - recall_patients

print(f"Normal patients     : {len(normal_patients)}")
print(f"Recall patients     : {len(recall_patients)}")
print(f"Overlap removed     : {len(normal_patients & recall_patients)}")
print(f"Qualifying patients : {len(qualifying_patients)}")

normal = cohort[cohort['PATIENT_STUDY_ID'].isin(qualifying_patients)].copy()
print(f"normal rows      : {len(normal)}")
print(f"Index breakdown:\n{normal['Index'].value_counts(dropna=False)}")

Normal patients     : 2
Recall patients     : 0
Overlap removed     : 0
Qualifying patients : 2
normal rows      : 13
Index breakdown:
Index
NaN        6
INDEX-1    3
INDEX      3
INDEX-2    1
Name: count, dtype: int64


In [66]:
cohort

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,SIDE,Slab,Series,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,Index,time_to_INDEX_days,time_to_INDEX_months
0,4333210516,77,72939576,2017-04-08,SCREEN,L,NaN,DBT,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2017-04-08,NaN,NaN,NaN
1,4333210516,77,72939576,2017-04-08,SCREEN,R,NaN,DBT,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2017-04-08,NaN,NaN,NaN
2,4333210516,78,78114012,2018-05-11,SCREEN,L,NaN,NaN,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-05-11,NaN,NaN,NaN
3,4333210516,78,78114012,2018-05-11,SCREEN,R,NaN,NaN,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-05-11,NaN,NaN,NaN
4,4333210516,79,64957868,2019-05-24,SCREEN,L,NaN,NaN,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-05-24,NaN,NaN,NaN
5,4333210516,79,64957868,2019-05-24,SCREEN,R,NaN,NaN,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-05-24,NaN,NaN,NaN
6,4333210516,81,66178393,2021-04-21,SCREEN,L,NaN,DBT,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2021-04-21,INDEX-1,593.0,19.5
7,4333210516,81,66178393,2021-04-21,SCREEN,R,NaN,DBT,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2021-04-21,INDEX-1,593.0,19.5
8,4333210516,83,459318559,2022-12-05,DIAG,L,Y,DBT,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2022-12-05,INDEX,NaN,NaN
9,4333210516,83,459318559,2022-12-05,DIAG,R,Y,DBT,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2022-12-05,INDEX,NaN,NaN


In [67]:
normal_cohort_candidate = cohort[cohort['PATIENT_STUDY_ID'].isin(qualifying_patients)].reset_index(drop=True)
normal_cohort_candidate.sort_values(['PATIENT_STUDY_ID', 'StudyDate', 'ACCESSION_NUMBER']).reset_index(drop=True, inplace=True)

normal_cohort_candidate["EXAM_COMPLETED_DATE"] = normal_cohort_candidate["EXAM_COMPLETED_DATE"].dt.strftime("%Y-%m-%d")

##### <span style="color:#FF6347;">**SAVE**</span> **normal_candidate_cohort**

In [68]:
normal_cohort_candidate['PATIENT_STUDY_ID'].nunique()

2

In [69]:
# output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'normal_cohort_candidate' + ".xlsx")
output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, 'normal_cohort_candidate' + ".xlsx")

normal_cohort_candidate.to_excel(output_file, index=False)